# 00: EEG Data Exploration

This notebook explores the EEG dataset and validates the data loading and preprocessing pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd() / 'src'))

from white_noise_experimentation.data.loaders import load_eeg_dataset, bandpass_filter, normalize_eeg
from white_noise_experimentation.config import load_config

print("Imports successful!")

## Load Dataset

In [ ]:
# Load data
data_loaders = load_eeg_dataset(
    data_root="./data/eeg_sample",
    window_size=256,
    window_stride=128,
    normalize=True,
    batch_size=64,
    num_workers=0,
)

train_loader = data_loaders['train']
val_loader = data_loaders['val']
test_loader = data_loaders['test']

print(f"Train set: {len(train_loader.dataset)} samples")
print(f"Val set: {len(val_loader.dataset)} samples")
print(f"Test set: {len(test_loader.dataset)} samples")

## Inspect Sample Data

In [ ]:
# Get a batch
x_batch, y_batch = next(iter(train_loader))

print(f"Batch shape: {x_batch.shape}")
print(f"Labels shape: {y_batch.shape}")
print(f"Unique labels: {torch.unique(y_batch)}")
print(f"Data dtype: {x_batch.dtype}")
print(f"Data range: [{x_batch.min():.4f}, {x_batch.max():.4f}]")

## Visualize EEG Signals

In [ ]:
# Visualize first few samples
n_samples_to_plot = 3
n_channels = 10  # Plot first 10 channels

fig, axes = plt.subplots(n_samples_to_plot, 1, figsize=(14, 10))

for i in range(n_samples_to_plot):
    sample = x_batch[i, :n_channels, :].numpy()
    
    # Plot first n_channels
    for ch in range(n_channels):
        axes[i].plot(sample[ch], alpha=0.6, label=f'Ch {ch}' if ch < 5 else '')
    
    axes[i].set_ylabel('Amplitude (z-scored)')
    axes[i].set_title(f'Sample {i} (first {n_channels} channels)')
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend(loc='upper right')

axes[-1].set_xlabel('Time (samples)')
plt.tight_layout()
plt.show()

print(f"Visualized {n_samples_to_plot} samples with {n_channels} channels each")

## Data Statistics

In [ ]:
# Compute statistics across all batches
all_data = []

for x, _ in train_loader:
    all_data.append(x.numpy())

all_data = np.concatenate(all_data, axis=0)
print(f"Total training data shape: {all_data.shape}")

print(f"\nStatistics across all training samples:")
print(f"  Mean: {all_data.mean():.6f}")
print(f"  Std: {all_data.std():.6f}")
print(f"  Min: {all_data.min():.6f}")
print(f"  Max: {all_data.max():.6f}")
print(f"  Q25: {np.percentile(all_data, 25):.6f}")
print(f"  Median: {np.percentile(all_data, 50):.6f}")
print(f"  Q75: {np.percentile(all_data, 75):.6f}")

## Load Configuration

In [ ]:
# Load a sample config
config = load_config("experiments/configs/ann_baseline.yaml")

print("Config loaded successfully!")
print(f"\nModel type: {config.model.type}")
print(f"Latent dim: {config.model.latent_dim}")
print(f"Learning rate: {config.train.lr}")
print(f"Epochs: {config.train.epochs}")
print(f"Noise enabled: {config.noise.enabled}")

## Test Model Creation

In [ ]:
from white_noise_experimentation.models.ann_autoencoder import ANNAutoencoder
from white_noise_experimentation.models.denoising_autoencoder import DenoisingAutoencoder

# Create ANN autoencoder
ann_ae = ANNAutoencoder(
    n_channels=64,
    window_size=256,
    latent_dim=64,
    hidden_dims=[128, 64]
)

# Create denoising autoencoder
denoise_ae = DenoisingAutoencoder(
    n_channels=64,
    window_size=256,
    latent_dim=64,
    hidden_dims=[128, 64],
    sigma=0.1
)

print(f"ANN Autoencoder created: {sum(p.numel() for p in ann_ae.parameters())} parameters")
print(f"Denoising Autoencoder created: {sum(p.numel() for p in denoise_ae.parameters())} parameters")

# Forward pass test
x_test = x_batch[:2]  # 2 samples
y_ann = ann_ae(x_test)
y_denoise = denoise_ae(x_test)

print(f"\nForward pass successful!")
print(f"Input shape: {x_test.shape}")
print(f"ANN output shape: {y_ann.shape}")
print(f"Denoising output shape: {y_denoise.shape}")